# Homework #4: Evaluation

## Environment setup

We provide an external file named `setup-environment.sh` to prepare the environment for this homework.

### Load documents from GitHub

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Number of loaded documents: {len(documents)}")

Number of loaded documents: 72


### Generating ground truth

To evaluate search, we need a dataset of questions where we know which document is the correct answer. This is the ground truth.

We want the output as a list of strings, so we define that structure with a Pydantic model, Then, we need to define a `Questions` data model:

In [10]:
from pydantic import BaseModel

# Define the expected structured output schema
class Questions(BaseModel):
    questions: list[str]

The module's instructions generate questions from a FAQ record, so we adapt them for a lesson page:

In [12]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [13]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

We'll need this pattern again in other evaluation sections today, so we put it in a reusable helper.

In [29]:
from evaluation_utils import llm_structured

## Question #1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

* 01-agentic-rag/lessons/01-intro.md
* 01-agentic-rag/lessons/02-environment.md
* 01-agentic-rag/lessons/03-rag.md

### Tracking cost

The response also contains token usage:

`usage.input_tokens, usage.output_tokens`

In [31]:
from evaluation_utils import calc_price

cost = calc_price(usage)
print(cost)

{'input_cost': 0.00131475, 'output_cost': 0.000495, 'total_cost': 0.00180975}


In [42]:
import json

total_input_tokens = 0
sample_documents = documents[:3]
for doc in sample_documents:
    print(len(doc["content"]), doc["filename"])
    user_prompt = json.dumps(doc)
    
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    # Extract prompt tokens from the response usage object
    input_tokens = usage.input_tokens
    total_input_tokens += input_tokens
    print(f"Document: {doc["filename"]} -> Input Tokens: {input_tokens}")

average_input_tokens = total_input_tokens / len(sample_documents)
print(f"\nAverage Input Tokens: {average_input_tokens}")

3183 01-agentic-rag/lessons/01-intro.md
Document: 01-agentic-rag/lessons/01-intro.md -> Input Tokens: 1020
3680 01-agentic-rag/lessons/02-environment.md
Document: 01-agentic-rag/lessons/02-environment.md -> Input Tokens: 1286
5920 01-agentic-rag/lessons/03-rag.md
Document: 01-agentic-rag/lessons/03-rag.md -> Input Tokens: 1753

Average Input Tokens: 1353.0


## Answer #1

> Average Input Tokens: 1353 approx. **1400**

### The full ground truth

You don't need to generate the data for the rest of the homework. We already did it for all 72 pages, using the same approach as in the lessons, and saved the 360 questions to a file.

Download it:

In [43]:
%%sh

PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget -N ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

--2026-07-13 16:59:13--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

     0K .......... .......... .......... .......... .......   100% 20.3M=0.002s

Last-modified header missing -- time-stamps turned off.
2026-07-13 16:59:13 (20.3 MB/s) - ‘ground-truth.csv’ saved [48627/48627]



Load it with pandas into a list of records called ground_truth. Each record has a question and the filename of the page that should answer it.

In [62]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
display(df_ground_truth.head(5))

ground_truth = df_ground_truth.to_dict(orient="records")

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


### Searching the chunks

We search over the same chunks as in homework 2.

In [63]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


Now rebuild the search from homework 2 over these chunks. Build a text index (Index) and a vector index (VectorSearch), both keyed on filename. Wrap each one in a function, text_search and vector_search, that takes a query and the number of results to return (5 by default).

In [64]:
from embedder import Embedder
from minsearch import Index, VectorSearch

tx_index = Index(text_fields=["content"])
tx_index.fit(chunks)

embedder = Embedder()

chunk_contents = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(chunk_contents)

vs_index = VectorSearch(keyword_fields=["filename"])
vs_index.fit(X, chunks)

Wrap each one in a function, text_search and vector_search, that takes a query and the number of results to return (5 by default).

In [66]:
def text_search(query, num_results=5):
    return tx_index.search(
        query,
        num_results=num_results
    )

print(text_search("certificate"))

[{'start': 2000, 'content': 'you want to receive a certificate, you need to submit your project while we\'re still accepting submissions.\n\nCourse: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nYou don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nWhat is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?\nThe zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.\n\nCloud alternatives with GPU\nCheck the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.\n"""\n```\n\nNotice the prompt doesn\'t end with `Answer:`. With older models like\nGPT-3 we added that to nudge the model into completing the s

In [71]:
def vector_search(query, num_results=5):
    vq = embedder.encode(query)
    return vs_index.search(
        query_vector=vq,
        num_results=num_results
    )

print(vector_search("certificate"))

[{'start': 2000, 'content': 'you want to receive a certificate, you need to submit your project while we\'re still accepting submissions.\n\nCourse: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nYou don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nWhat is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?\nThe zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.\n\nCloud alternatives with GPU\nCheck the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.\n"""\n```\n\nNotice the prompt doesn\'t end with `Answer:`. With older models like\nGPT-3 we added that to nudge the model into completing the s

For hybrid search, reuse the `rrf` function from homework 2:

In [72]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Then define `hybrid_search` on top of it:

In [73]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

print(hybrid_search("certificate"))

[{'start': 2000, 'content': 'you want to receive a certificate, you need to submit your project while we\'re still accepting submissions.\n\nCourse: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nYou don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nWhat is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?\nThe zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.\n\nCloud alternatives with GPU\nCheck the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.\n"""\n```\n\nNotice the prompt doesn\'t end with `Answer:`. With older models like\nGPT-3 we added that to nudge the model into completing the s

## Question #2. First result with text search

Take the first question from the ground truth:

In [79]:
query = ground_truth[0]["question"]
results = text_search(query, num_results=1)
print(f"The first result is: {results[0]["filename"]}")

The first result is: 01-agentic-rag/lessons/03-rag.md


## Answer #2

> The first result is: **01-agentic-rag/lessons/03-rag.md**

Question #3. First result with vector search

After running vector_search for the same question, what's the filename of the first result?

In [80]:
query = ground_truth[0]["question"]
results = vector_search(query, num_results=1)
print(f"The first result is: {results[0]["filename"]}")

The first result is: 01-agentic-rag/lessons/01-intro.md


## Answer #3

> The first result from vector search: **01-agentic-rag/lessons/01-intro.md**